# DSA 495 · Module 05
## Text Classification III: Evaluation and Error Analysis

Classify tweets as **negative, neutral, or positive** using RoBERTa and BART. Connect their predictions to precision, recall, F1, and confusion matrices.

**Workflow:** understand the metrics → load the data → evaluate fixed RoBERTa → develop BART label wording → freeze the wording → evaluate BART → compare both models.

Both models are already trained. This notebook runs inference; it does not update model weights.

## 0 · Setup

In **Google Colab**, select a T4 GPU if available and upload these three files from the module's `data` folder through the Files sidebar:

- `illustrative_metrics_12.csv`
- `tweeteval_sentiment_development_60.csv`
- `tweeteval_sentiment_evaluation_300.csv`

The code uses `DATA_DIR = Path("/content")`. For a local run from the `notebooks` folder, change it to `Path("../data")`. Model downloads require internet; CPU inference is slower. Run the cells in order.

In [ ]:
%pip install -q "torch>=2.6,<3" "transformers==4.57.6" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4"

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score,
)
from transformers import AutoTokenizer, pipeline

In [ ]:
DATA_DIR = Path("/content")  # Colab uploads; use Path("../data") when running locally.
LABELS = ["negative", "neutral", "positive"]
DEVICE = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 8

torch.manual_seed(495)
torch.set_num_threads(4)
pd.set_option("display.max_colwidth", 180)
print("Device:", DEVICE, "(0 = CUDA GPU; -1 = CPU)")

## 1 · From texts to counts

These 12 texts and predictions are **invented teaching examples**, not measured model results. Here, **neutral** is the class of interest.

| Count | Reference label | Predicted label |
|---|---|---|
| TP | neutral | neutral |
| FP | another class | neutral |
| FN | neutral | another class |
| TN | another class | another class |

“True positive” means a correct prediction of the class of interest; it does not necessarily mean positive sentiment.

**Check:** Find one example of each count before running the next calculation.

In [ ]:
practice = pd.read_csv(DATA_DIR / "illustrative_metrics_12.csv")
practice

In [ ]:
# For this question, neutral is the class of interest.
reference_neutral = practice["label"].eq("neutral")
predicted_neutral = practice["prediction"].eq("neutral")

true_positives = practice[reference_neutral & predicted_neutral]
false_positives = practice[~reference_neutral & predicted_neutral]
false_negatives = practice[reference_neutral & ~predicted_neutral]
true_negatives = practice[~reference_neutral & ~predicted_neutral]

print("TP:", true_positives["example_id"].tolist())
print("FP:", false_positives["example_id"].tolist())
print("FN:", false_negatives["example_id"].tolist())
print("TN:", true_negatives["example_id"].tolist())

### Precision, recall, and F1

**Precision:** Of the texts predicted neutral, how many have a neutral reference label?

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

**Recall:** Of the reference-neutral texts, how many did the model find?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

**F1:** The harmonic mean of precision and recall.

$$
F1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2TP}{2TP + FP + FN}
$$

F1 excludes true negatives and is not the arithmetic average. Error types are relative to the class being measured.

In [ ]:
tp = len(true_positives)
fp = len(false_positives)
fn = len(false_negatives)

neutral_precision = tp / (tp + fp)
neutral_recall = tp / (tp + fn)
neutral_f1 = 2 * tp / (2 * tp + fp + fn)

pd.Series({
    "precision": neutral_precision, "recall": neutral_recall, "F1": neutral_f1,
})

## 2 · Two datasets, two purposes

The real data are English tweets from **TweetEval sentiment**. Reference labels are annotations; sarcasm, mixed sentiment, and missing context can make a label debatable.

| Dataset | Size | Purpose in this notebook |
|---|---:|---|
| Development | 60 tweets; 20 per class | Compare BART wording, inspect errors, and choose a formulation. |
| Evaluation | 300 tweets; uneven class counts | Measure fixed RoBERTa and BART with the selected wording on the same examples. |

Development comes from the official validation split; evaluation comes from the official test split. Preparation used seed 495 and checked duplicate texts and split overlap. The two classroom sets have distinct IDs and normalized texts; these checks cannot rule out paraphrases or prior model exposure.

**Development changes our wording choice. Evaluation measures that fixed choice.** Evaluation results are not used to revise BART wording.

Data source: [TweetEval](https://github.com/cardiffnlp/tweeteval), sentiment subset (CC BY 3.0; Twitter terms also apply). See the module's [data notes](../data/README.md) for preparation and attribution.

In [ ]:
development = pd.read_csv(DATA_DIR / "tweeteval_sentiment_development_60.csv")
evaluation = pd.read_csv(DATA_DIR / "tweeteval_sentiment_evaluation_300.csv")

class_counts = pd.DataFrame({  # Create a table of class counts for both datasets.
    "development": development["label"].value_counts(),  # Count development tweets for each sentiment.
    "evaluation": evaluation["label"].value_counts(),  # Count evaluation tweets for each sentiment.
}).reindex(LABELS).fillna(0).astype(int)  # Order rows by LABELS, replace missing counts with zero, and use integers.
class_counts.index.name = "label"  # Name the row index "label".

display(class_counts)  # Display the class-count table.

## 3 · RoBERTa: text → tokens → scores → predictions

[RoBERTa sentiment](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment) was already fine-tuned for this task. Its labels map to negative, neutral, and positive. We use its fixed checkpoint without selecting wording.

The original `text` stays unchanged. Both models receive `model_text`, where usernames become `@user` and URLs become `http`.

In [ ]:
ENCODER_ID = "cardiffnlp/twitter-roberta-base-sentiment"
ENCODER_REVISION = "daefdd1f6ae931839bce4d0f3db0a1a4265cd50f"
BART_ID = "facebook/bart-large-mnli"
BART_REVISION = "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce"

encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER_ID, revision=ENCODER_REVISION)
bart_tokenizer = AutoTokenizer.from_pretrained(BART_ID, revision=BART_REVISION)

In [ ]:
# Match the model card's username/URL convention, retaining the original text.
development["model_text"] = development["text"].str.replace(r"(?<!\S)@\S+", "@user", regex=True)
development["model_text"] = development["model_text"].str.replace(r"(?<!\S)http\S*", "http", regex=True)
evaluation["model_text"] = evaluation["text"].str.replace(r"(?<!\S)@\S+", "@user", regex=True)
evaluation["model_text"] = evaluation["model_text"].str.replace(r"(?<!\S)http\S*", "http", regex=True)

example_text = development.loc[0, "model_text"]  # Select the processed development text at index 0.
print(example_text)  # Print the text used for the tokenization example.

In [ ]:
content_ids = encoder_tokenizer.encode(example_text, add_special_tokens=False)  # Tokenize the text without special tokens.
all_ids = encoder_tokenizer.encode(example_text, add_special_tokens=True)  # Tokenize the text with special tokens.

token_table = pd.DataFrame([{  # Create a table containing RoBERTa's tokenization results.
    "tokenizer": "RoBERTa",  # Record the tokenizer name.
    "tokens": encoder_tokenizer.convert_ids_to_tokens(content_ids),  # Convert content IDs to vocabulary token strings.
    "token_ids": content_ids,  # Record the content token IDs.
    "content_length": len(content_ids),  # Count tokens without special tokens.
    "length_with_special_tokens": len(all_ids),  # Count tokens including special tokens.
}])  # Finish creating the table.

with pd.option_context("display.max_colwidth", None):  # Temporarily display full cell contents without truncation.
    display(token_table)  # Display RoBERTa's tokenization results.

**Check:** Find a word split into multiple tokens. How many extra tokens are added when special tokens are included?

### RoBERTa evaluation · 300 tweets

The fixed classifier processes the evaluation tweets. Each tweet receives three sentiment scores; the highest-scoring label becomes its prediction. A high score does not guarantee correctness.

In [ ]:
encoder = pipeline(
    "text-classification", model=ENCODER_ID, revision=ENCODER_REVISION,
    tokenizer=encoder_tokenizer, device=DEVICE,
)
print(encoder.model.config.id2label)
ENCODER_LABEL_MAP = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

In [ ]:
start = time.perf_counter()  # Record the starting time.
raw_encoder_predictions = encoder(  # Run RoBERTa sentiment classification.
    evaluation["model_text"].tolist(), top_k=None, batch_size=BATCH_SIZE,  # Process evaluation texts in batches and return scores for all labels.
    truncation=True, max_length=512,  # Truncate inputs to at most 512 tokens, including special tokens.
)  # Store the prediction outputs.
encoder_seconds = time.perf_counter() - start  # Calculate elapsed inference time in seconds.

In [ ]:
raw_encoder_predictions[0]  # Show all sentiment labels and scores for the first evaluation text.

### Scores → labels → report

`idxmax(axis=1)` selects the label with the largest score. `max(axis=1)` returns that score. Reference labels remain separate from model predictions.

In [ ]:
results = evaluation.copy()  # Copy evaluation data to store predictions.

encoder_rows = []
for prediction in raw_encoder_predictions:  # Collect sentiment scores for each tweet.
    row = {ENCODER_LABEL_MAP[item["label"]]: item["score"] for item in prediction}
    encoder_rows.append(row)

encoder_scores = pd.DataFrame(encoder_rows)[LABELS]  # Create one score column per sentiment.
results["encoder_prediction"] = encoder_scores.idxmax(axis=1)  # Select the highest-scoring sentiment.
results["encoder_score"] = encoder_scores.max(axis=1)  # Store its score.

display(results[["text", "label", "encoder_prediction", "encoder_score"]].head())

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    results["label"],  # True labels appear on the rows.
    results["encoder_prediction"],  # Predicted labels appear on the columns.
    labels=LABELS,  # Keep the same class order as the classification report.
    display_labels=LABELS,  # Label both axes with sentiment names.
    cmap="Blues",  # Use darker blue for larger counts.
    colorbar=False,  # Hide the color scale.
    values_format="d",  # Display counts as integers.
)
plt.title("RoBERTa sentiment classification")
plt.xlabel("Predicted sentiment")
plt.ylabel("Reference sentiment (Ground Truth)")
plt.grid(False)
plt.show()

In [ ]:
print(classification_report(  # Show precision, recall, F1, and support.
    results["label"],
    results["encoder_prediction"],
    labels=LABELS,
    digits=3,
    zero_division=0,
))

### Check the confusion matrix

Rows are reference labels; columns are predictions. For one class, the diagonal is TP, the rest of its column is FP, and the rest of its row is FN. Its row total is its support.

**Your turn:** Find the class with the lowest recall. Calculate its precision, recall, and F1 from the counts; round final answers to three decimals. Explain why precision and recall use different denominators.

## 4 · BART zero-shot development · 60 tweets

[BART-MNLI](https://huggingface.co/facebook/bart-large-mnli) compares each tweet with candidate statements such as “The sentiment of this text is positive.” It returns descriptions ranked by score; we map them back to sentiment names.

**Development pipeline:** candidate wording A and B → predictions on all 60 development tweets → development metrics and errors → wording choice.

Only the label descriptions change. The model, hypothesis template, and development tweets stay fixed. With `multi_label=False`, the candidate scores sum to one; they depend on the supplied candidates.

**Selection rule:** highest development macro-F1; A wins an exact tie. Macro-F1 gives each class equal weight. This is wording selection, not model training.

In [ ]:
zero_shot = pipeline(  # Create a zero-shot classification pipeline.
    "zero-shot-classification", model=BART_ID, revision=BART_REVISION,  # Load the pinned BART checkpoint for zero-shot classification.
    tokenizer=bart_tokenizer, device=DEVICE,  # Use the BART tokenizer and the selected GPU or CPU.
)  # Finish creating the pipeline.

FORMULATIONS = {
    "A: sentiment names": {"negative": "negative", "neutral": "neutral", "positive": "positive"},
    "B: descriptions": {
        "negative": "negative, expressing criticism, dissatisfaction, or disappointment",
        "neutral": "neutral, giving information without a clear positive or negative opinion",
        "positive": "positive, expressing praise, satisfaction, or enthusiasm",
    },
}
HYPOTHESIS_TEMPLATE = "The sentiment of this text is {}."

print("Text:", development.loc[0, "model_text"])  # Display the processed development text at index 0.
for description in FORMULATIONS["A: sentiment names"].values():  # Loop through the three sentiment names in formulation A.
    print("Hypothesis:", HYPOTHESIS_TEMPLATE.format(description))  # Insert each sentiment into the template and print the hypothesis.

### Compare wording on development data

The next loop calls the **same BART model once per formulation** on the same 60 development tweets. It does not call RoBERTa or evaluate on the 300-tweet set.

In [ ]:
development_predictions = development[["example_id", "text", "label"]].copy()  # Copy IDs, original texts, and reference labels to store predictions.
development_rows = []  # Create a list to store metrics for each formulation.
raw_development_predictions = {}  # Create a dictionary to store each formulation's full model outputs.

In [ ]:
for formulation_name, label_descriptions in FORMULATIONS.items():  # Loop through both label formulations.
    outputs = zero_shot(  # Run BART zero-shot classification.
        development["model_text"].tolist(),  # Supply the processed development texts.
        candidate_labels=list(label_descriptions.values()),  # Use this formulation's descriptions as candidate labels.
        hypothesis_template=HYPOTHESIS_TEMPLATE, multi_label=False, batch_size=BATCH_SIZE,  # Apply the template and score competing labels in batches.
    )  # Finish the classification call.
    raw_development_predictions[formulation_name] = outputs  # Save the full outputs for this formulation.
    description_to_label = {description: label for label, description in label_descriptions.items()}  # Map descriptions back to standard sentiment names.
    predictions = [description_to_label[output["labels"][0]] for output in outputs]  # Get each text's highest-scoring label and convert it to a sentiment name.
    development_predictions[formulation_name] = predictions  # Store predictions in a column named after the formulation.
    development_rows.append({  # Add this formulation's evaluation metrics.
        "formulation": formulation_name,  # Record the formulation name.
        "accuracy": accuracy_score(development["label"], predictions),  # Calculate the fraction of predictions that are correct.
        "macro_f1": f1_score(development["label"], predictions, labels=LABELS, average="macro"),  # Calculate the unweighted mean of the three class F1 scores.
    })  # Finish adding the metrics.

development_comparison = pd.DataFrame(development_rows).set_index("formulation")  # Create a comparison table with formulation names as row labels.
development_comparison.round(3)  # Display the metrics rounded to three decimal places.

### Inspect development disagreements

The table shows where A and B disagree. It excludes errors they share. A wording revision can fix some errors and introduce others, so compare performance across **all** development examples.

In [ ]:
changed = development_predictions[  # Select development tweets where A and B disagree.
    development_predictions["A: sentiment names"] != development_predictions["B: descriptions"]  # Compare predicted sentiments.
]  # Keep only changed predictions.
changed  # Display reference labels alongside both predictions.

### Optional development exercise · revised wording C

Could broader positive wording and a clearer neutral description help? This is a hypothesis to test on development data.

```python
FORMULATIONS["C: broader sentiment descriptions"] = {
    "negative": "negative, expressing an unfavorable opinion, criticism, dissatisfaction, or disappointment",
    "neutral": "neutral, reporting, announcing, or quoting information without expressing a clear favorable or unfavorable attitude",
    "positive": "positive, expressing a favorable opinion, approval, enjoyment, admiration, or excitement",
}
```

To try C, add it before the development loop, then rerun the development initialization and comparison cells. Keep the template fixed. Inspect both corrected and newly introduced errors across all 60 tweets. The selection cell below uses the highest development macro-F1 among the formulations tested; an exact tie selects the first listed formulation.

Complete any wording experiments **before BART evaluation**. If you skip C, the notebook compares A and B only.

### Freeze the wording

Select the wording using development macro-F1, even if a different formulation has higher accuracy. A small development-score difference does not establish a general advantage.

In [ ]:
# idxmax returns the first maximum; A is first for the prespecified tie rule.
selected_formulation = development_comparison["macro_f1"].idxmax()  # Select the highest development macro-F1.
selected_descriptions = FORMULATIONS[selected_formulation]  # Retrieve the chosen sentiment descriptions.
print("Frozen choice:", selected_formulation)  # Fix the wording before evaluation.

## 5 · BART final evaluation · 300 tweets

**Evaluation pipeline:** frozen wording → one BART run on the 300 evaluation tweets → predictions → classification report.

We run BART again because these are different tweets. Only the selected wording is used. This tests that fixed choice on the same evaluation examples used for RoBERTa.

Keep the wording fixed after viewing these results. Evaluation errors describe limitations; revising wording from those errors would require fresh held-out data for a new final assessment.

In [ ]:
start = time.perf_counter()  # Record the starting time.

raw_bart_predictions = zero_shot(  # Run BART zero-shot classification on the evaluation set.
    evaluation["model_text"].tolist(),  # Supply the processed evaluation tweets as a list.
    candidate_labels=list(selected_descriptions.values()),  # Use the label descriptions selected during development.
    hypothesis_template=HYPOTHESIS_TEMPLATE, multi_label=False, batch_size=BATCH_SIZE,  # Apply the template and score competing sentiments in batches.
)  # Store the predicted descriptions and their scores for each tweet.

bart_seconds = time.perf_counter() - start  # Calculate elapsed inference time in seconds.

In [ ]:
raw_bart_predictions[0]

In [ ]:
description_to_label = {  # Map descriptions back to standard sentiment names.
    description: label for label, description in selected_descriptions.items()
}

results["bart_prediction"] = [  # Store the highest-scoring sentiment for each tweet.
    description_to_label[prediction["labels"][0]]
    for prediction in raw_bart_predictions
]
results["bart_score"] = [  # Store the score of each predicted sentiment.
    prediction["scores"][0]
    for prediction in raw_bart_predictions
]

print(classification_report(  # Display precision, recall, F1, and support.
    results["label"],
    results["bart_prediction"],
    labels=LABELS,
    digits=3,
    zero_division=0,
))

## 6 · Compare the two models

Both models are measured on the **same 300 evaluation tweets**.

| Metric | Meaning |
|---|---|
| Accuracy | Fraction of all predictions that match the reference labels. |
| Macro-F1 | Average of the class F1 scores, with equal weight per class. |
| Weighted-F1 | Average of the class F1 scores, weighted by reference support. |

Inference time excludes model loading and BART development runs. Hardware and batch size affect it; BART evaluates three hypotheses per tweet. The models have different training histories, so this compares the two workflows rather than isolating an architecture effect.

In [ ]:
METHOD_COLUMNS = {
    "RoBERTa sentiment": "encoder_prediction",
    "BART zero-shot": "bart_prediction",
}
inference_seconds = {
    "RoBERTa sentiment": encoder_seconds,
    "BART zero-shot": bart_seconds,
}

comparison_rows = []
for method, column in METHOD_COLUMNS.items():
    comparison_rows.append({
        "method": method,
        "accuracy": accuracy_score(results["label"], results[column]),
        "macro_f1": f1_score(
            results["label"], results[column],
            labels=LABELS, average="macro", zero_division=0,
        ),
        "weighted_f1": f1_score(
            results["label"], results[column],
            labels=LABELS, average="weighted", zero_division=0,
        ),
        "inference_seconds": inference_seconds[method],
    })

comparison = pd.DataFrame(comparison_rows).set_index("method")
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for (method, column), ax in zip(METHOD_COLUMNS.items(), axes):
    ConfusionMatrixDisplay.from_predictions(
        results["label"], results[column], labels=LABELS,
        cmap="Blues", colorbar=False, values_format="d", ax=ax,
    )
    ax.set_title(method)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Reference label")

fig.suptitle("Same 300 evaluation tweets · counts, not percentages")
plt.tight_layout()
plt.show()

### Read the results

Which model has the higher macro-F1? For BART's neutral class, compare the false positives and false negatives. How can precision be high while recall is low?

Use class-level errors alongside overall metrics when discussing a model's strengths and limitations. These results describe this sample, not all tweets.

References: [classification reports](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) · [data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).